# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL and follows the FAIR Principles.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_obj = dataset.metadata

print(f"{metadata_obj.name}: {metadata_obj.description}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We will enumerate the record sets and their fields, listing all unique `@id` values (these are essential for referencing data in the Croissant schema and when extracting data below).

In [ ]:
# List all record sets and their field @ids
record_sets = dataset.metadata.record_sets
print(f"Number of record sets in dataset: {len(record_sets)}")
record_set_overview = []
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    print(f"  Name: {rs.name}")
    print(f"  Description: {getattr(rs, 'description', '')}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - {field.id} (name: {field.name}, type: {field.data_type})")
    record_set_overview.append({'id': rs.id, 'name': rs.name, 'fields': [f.id for f in rs.fields]})
    print('----')
    
if not record_sets:
    print('No record sets found in the Croissant metadata. Please check the schema.')

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from above.

Usually, Croissant datasets with tabular data have one main RecordSet for tables, which references all actual columns by `@id`.

We'll attempt to load all record sets into pandas DataFrames, referencing RecordSets and their fields strictly by their `@id`.

In [ ]:
# Extract data from all record sets using their @id
all_record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

# We'll print available record set ids
print("Available RecordSet @ids:")
print(all_record_set_ids)

if not all_record_set_ids:
    print("No record sets available to extract data from.")
else:
    for rs_id in all_record_set_ids:
        print(f"Loading records for RecordSet @id: {rs_id}")
        records = list(dataset.records(record_set=rs_id))
        df = pd.DataFrame(records)
        dataframes[rs_id] = df
        print(f"DataFrame shape for {rs_id}: {df.shape}")
    # For demonstration, pick the first record set
    first_rs_id = all_record_set_ids[0]
    print(f"\nColumns in DataFrame for RecordSet @id '{first_rs_id}':")
    print(dataframes[first_rs_id].columns.tolist())
    dataframes[first_rs_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

We'll pick a numeric field from the first record set and filter, normalize, and group by a categorical variable if possible. All field and group references are by their `@id`.

In [ ]:
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Select the first RecordSet for demonstration
record_set_id = None
if dataframes:
    record_set_id = list(dataframes.keys())[0]
    df = dataframes[record_set_id]
    print(f"Using RecordSet @id: {record_set_id}")
    # List all fields available
    field_ids = df.columns.tolist()
    print("Available field @ids in this record set:")
    print(field_ids)
    # Try to guess a numeric field (float or integer)
    candidate_numeric_field = None
    for col in field_ids:
        # Try convert to numeric
        try:
            if np.issubdtype(df[col].dropna().astype(float).dtype, np.number):
                candidate_numeric_field = col
                break
        except Exception:
            continue
    if candidate_numeric_field is not None:
        numeric_field = candidate_numeric_field
        # Filtering records with value > threshold (e.g. 10.0)
        try:
            numeric_vals = pd.to_numeric(df[numeric_field], errors='coerce')
            threshold = 10
            filtered_df = df[numeric_vals > threshold].copy()
            print(f"Filtered records with {numeric_field} > {threshold}:")
            print(filtered_df.head())

            # Normalization
            filtered_df[f"{numeric_field}_normalized"] = (
                pd.to_numeric(filtered_df[numeric_field], errors='coerce') - numeric_vals.mean()
            ) / numeric_vals.std()
            print(f"Normalized {numeric_field} for filtered records:")
            print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

            # Try grouping by a (first available) categorical field
            group_field = None
            for col in field_ids:
                if col != numeric_field:
                    # If not mostly numeric, treat as group field
                    try:
                        # If most values are not null and cannot be converted to float, treat as categorical
                        test_s = df[col].dropna()[:20]
                        non_numeric = sum(pd.to_numeric(test_s, errors='coerce').isna())
                        if non_numeric > len(test_s) / 2:
                            group_field = col
                            break
                    except Exception:
                        group_field = col
                        break
            if group_field and group_field in filtered_df.columns:
                grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
                print(f"\nGrouped mean of '{numeric_field}' by '{group_field}':")
                print(grouped_df.head())
                # Save for visualization
                last_grouped_df = grouped_df
                last_group_field = group_field
            else:
                print("No suitable group (categorical) field found for grouping.")
        except Exception as e:
            print(f"Error in filtering or normalization: {e}")
    else:
        print("No numeric field found in first RecordSet.")
else:
    print("No DataFrame available for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset. We use the normalized numeric field or show grouped means if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# If previous section produced a filtered_df and normalized values
if 'filtered_df' in locals() and numeric_field in filtered_df:
    plt.figure(figsize=(7,4))
    sns.histplot(pd.to_numeric(filtered_df[numeric_field], errors='coerce').dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field} (filtered > {threshold})')
    plt.xlabel(numeric_field)
    plt.show()

    if f"{numeric_field}_normalized" in filtered_df:
        plt.figure(figsize=(7,4))
        sns.histplot(filtered_df[f"{numeric_field}_normalized"].dropna(), bins=30, kde=True)
        plt.title(f'Normalized {numeric_field} distribution (filtered)')
        plt.xlabel(f'{numeric_field}_normalized')
        plt.show()

    if 'last_grouped_df' in locals() and last_grouped_df.shape[0] > 1:
        plt.figure(figsize=(8,4))
        sns.barplot(data=last_grouped_df, x=last_group_field, y=numeric_field)
        plt.title(f'Mean {numeric_field} by {last_group_field}')
        plt.ylabel(f'Mean {numeric_field}')
        plt.xlabel(last_group_field)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No filtered numeric field for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to access and explore the FAIR^2 mass spectrometry dataset, referencing all entities by their Croissant `@id`s, as required for robust schema-based workflows.

You can reuse this notebook for any other Croissant-formatted dataset by updating only the schema URL and repeating the RecordSet/field review steps.

- All data selections, groupings, and transformations above use only `@id` references, ensuring reproducibility and schema compliance.
- For additional domain-specific analysis, use the available fields listed above and reference them by `@id`.

**If you encountered any issues:** Please verify that the Croissant schema contains at least one RecordSet with accessible tabular data; some datasets may require custom extraction logic for non-tabular content.